# Tokenizers

Explore how language-model tokenizers convert text and chat messages into token IDs.

The notebook covers:
- Encoding and decoding text
- Comparing tokenizer vocabularies
- Formatting chat prompts
- Comparing tokenization across model families

**Environment:** Python, Hugging Face Transformers, and access to the gated Meta Llama 3.1 model.

## Environment

This notebook can run in Google Colab or a local Python environment. A GPU is not required for tokenizer operations.

If a Colab runtime becomes unavailable, reconnect to a fresh runtime and rerun the notebook from the installation cell.

In [ ]:
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

In [ ]:
import os

from transformers import AutoTokenizer
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "Set HF_TOKEN in Colab Secrets or as a local environment variable."
    )

login(token=hf_token)

## Hugging Face authentication

1. Create an access token in [Hugging Face settings](https://huggingface.co/settings/tokens).
2. In Google Colab, add the token to Secrets as `HF_TOKEN`.
3. Grant the token read access to the models used in this notebook. Write access is not required for downloading model tokenizers.
4. Run the authentication cell below.

For local execution, set the `HF_TOKEN` environment variable before starting Jupyter.

In [ ]:
# Optional: inspect the available GPU in Google Colab.
try:
    import subprocess

    result = subprocess.run(
        ["nvidia-smi"],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        print(result.stdout)
    else:
        print("No NVIDIA GPU detected. Tokenizer operations can run on CPU.")
except FileNotFoundError:
    print("nvidia-smi is unavailable. Continuing without GPU diagnostics.")

## Meta Llama 3.1 access

The Meta Llama 3.1 model is gated. Request access on the [model page](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B) and accept the model terms before loading it.

The account associated with `HF_TOKEN` must have access to the model.

In [ ]:
# Load the base Llama 3.1 tokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B",
    trust_remote_code=True,
)

In [ ]:
text = "I am excited to show tokenizers in action to my LLM engineers"
tokens = tokenizer.encode(text)
tokens

In [ ]:
character_count = len(text)
word_count = len(text.split())
token_count = len(tokens)

print(
    f"Characters: {character_count} | "
    f"Words: {word_count} | "
    f"Tokens: {token_count}"
)

In [ ]:
tokenizer.decode(tokens)

In [ ]:
tokenizer.batch_decode(tokens)

In [ ]:
tokenizer.get_added_vocab()

In [ ]:
len(tokenizer.vocab)

## Instruction-tuned models and chat templates

Instruction-tuned models expect prompts in a model-specific chat format. Hugging Face tokenizers expose `apply_chat_template` to convert a list of role/content messages into the format expected by each model.

The method can return the formatted prompt as text or tokenize it directly.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {
        "role": "user",
        "content": "Tell a light-hearted joke for a room of data scientists.",
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print(prompt)

## From chat messages to model input

Chat messages are structured data at the application layer. Before they are passed to a language model, they are serialized into a model-specific prompt and tokenized.

The input pipeline is:

1. **Messages:** system, user, and assistant roles with their content.
2. **Chat template:** model-specific markers are added to represent message boundaries and control flow.
3. **Tokenization:** the formatted text is split into tokens.
4. **Token IDs:** each token is mapped to an integer in the tokenizer vocabulary.

The model consumes a sequence of token IDs and predicts a probability distribution over the next token.

## Compare tokenizer implementations

Compare tokenization and chat templates across the following instruction-tuned models:

- Phi-4 Mini (Microsoft)
- DeepSeek-V3.1
- Qwen2.5-Coder-7B

In [ ]:
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"

In [ ]:
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

text = "I am excited to explore Hugging Face tokenizers with LLM engineers."

for model_name, model_tokenizer in [
    ("Llama 3.1", tokenizer),
    ("Phi-4 Mini", phi4_tokenizer),
]:
    token_ids = model_tokenizer.encode(text)
    print(f"{model_name}:")
    print(token_ids)
    print(model_tokenizer.batch_decode(token_ids))
    print()

In [ ]:
for model_name, model_tokenizer in [
    ("Llama 3.1", tokenizer),
    ("Phi-4 Mini", phi4_tokenizer),
]:
    formatted_prompt = model_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    print(f"{model_name}:")
    print(formatted_prompt)
    print()

In [ ]:
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

text = "I am excited to explore Hugging Face tokenizers with LLM engineers."

for model_name, model_tokenizer in [
    ("Llama 3.1", tokenizer),
    ("Phi-4 Mini", phi4_tokenizer),
    ("DeepSeek-V3.1", deepseek_tokenizer),
]:
    print(f"{model_name}: {model_tokenizer.encode(text)}")

In [ ]:
for model_name, model_tokenizer in [
    ("Llama 3.1", tokenizer),
    ("Phi-4 Mini", phi4_tokenizer),
    ("DeepSeek-V3.1", deepseek_tokenizer),
]:
    formatted_prompt = model_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    print(f"{model_name}:")
    print(formatted_prompt)
    print()

In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)

code = """\
def hello_world(person):
    print("Hello", person)
"""

token_ids = qwen_tokenizer.encode(code)

for token_id in token_ids:
    print(f"{token_id}: {qwen_tokenizer.decode([token_id])}")